In [1]:
import numpy as np
import random
import os
import sys
import deepRD
import torch
from deepRD.diffusionIntegrators import langevinNoiseSampler
from deepRD.potentials import bistable
from deepRD.noiseSampler import noiseSampler
import deepRD.tools.trajectoryTools as trajectoryTools
import deepRD.tools.analysisTools as analysisTools
import multiprocessing
from multiprocessing import Pool
from functools import partial
import numpy as np
from tqdm import tqdm

In [2]:
def apply_periodic(q, boxsize):
    """
    Apply periodic boundary conditions to positions q.

    q       : [..., 3] (torch.Tensor)
    boxsize : float or array-like of length 3
              box spans [-L/2, L/2] in each dimension
    """
    if not torch.is_tensor(q):
        q = torch.as_tensor(q, dtype=torch.float32)

    if not torch.is_tensor(boxsize):
        boxsize = torch.as_tensor(boxsize, dtype=q.dtype, device=q.device)

    if boxsize.ndim == 0:
        boxsize = boxsize.expand(3)

    # Map to [0, L), then shift back to [-L/2, L/2)
    q_shifted = q + boxsize / 2.0
    q_wrapped = torch.remainder(q_shifted, boxsize)
    q_periodic = q_wrapped - boxsize / 2.0
    return q_periodic


def bistable_force(q, minimaDist, kconstants, scale=1.0):
    """
    Vectorised bistable force for positions q.

    q         : [..., 3]  (any leading batch dims, torch.Tensor)
    minimaDist: float
    kconstants: (3,) array-like or tensor [kx, ky, kz]
    scale     : float

    Returns:
        force: [..., 3]
    """
    if not torch.is_tensor(q):
        q = torch.as_tensor(q, dtype=torch.float32)

    kx, ky, kz = kconstants
    x = q[..., 0]
    y = q[..., 1]
    z = q[..., 2]

    force = torch.zeros_like(q)
    force[..., 0] = - kx * 4 * x * (x**2 - minimaDist**2) / (minimaDist**4)
    force[..., 1] = - ky * 2 * y
    force[..., 2] = - kz * 2 * z

    return scale * force


def aboba_deterministic_step(q_n, v_n, dt_eff, Gamma, mass,
                             minimaDist, kconstants, boxsize, scale=1.0):
    """
    Deterministic ABOBA step (no noise) for one time step dt_eff.

    q_n, v_n : [..., 3]
    dt_eff   : float (effective dt = step * dt)
    Gamma    : friction scalar
    mass     : mass scalar
    """
    # A: first half-step in position
    q_half = q_n + v_n * (dt_eff / 2.0)
    #q_half = apply_periodic(q_half, boxsize)

    # Force at x^{n+1/2}
    F_half = bistable_force(q_half, minimaDist, kconstants, scale=scale)

    # BOB step without noise:
    # expterm = exp(-Gamma * dt / mass)
    # frictionForceTerm = v_n * expterm + (1 + expterm) * F * dt/(2m)
    expterm = torch.exp(torch.tensor(-Gamma * dt_eff / mass,
                                     dtype=q_n.dtype, device=q_n.device))

    v_det = v_n * expterm + (1.0 + expterm) * F_half * (dt_eff / (2.0 * mass))

    # A: second half-step in position
    q_next = q_half + v_det * (dt_eff / 2.0)
    q_next = apply_periodic(q_next, boxsize)

    return q_next, v_det

In [3]:
def compute_r_cg_from_fine(q, v, k, parameters,
                           boxsize=5.0, scale=1.0):
    """
    Compute coarse-grained interaction noise r_cg from fine benchmark trajectories,
    using the same ABOBA scheme as langevinNoiseSampler but *without* noise.

    q, v : [n_traj, T, 3]  (fine dt)
    dt   : fine timestep
    k : integer k (dt_eff = k * dt)

    Gamma, mass       : friction and mass
    minimaDist        : bistable parameter
    kconstants        : (3,) for bistable (kx, ky, kz)
    boxsize           : scalar or (3,) – periodic box, interval [-L/2, L/2]
    scale             : scale factor for the potential

    Returns:
        q_next, v_next, r_next  : [n_traj, T-step, 3]

            v_next = v_det(q_n, v_n; dt_eff) + r_next
    """
    assert q.shape == v.shape
    assert q.ndim == 3 and q.shape[-1] == 3, "q, v must be [n_traj, T, 3]"
    assert k >= 1
    
    dt = parameters['dt']
    Gamma = parameters['Gamma']
    mass = parameters['mass']

    n_traj, T, _ = q.shape
    
    T_cg = T - k
    if T_cg <= 0:
        raise ValueError(f"Not enough timesteps T={T} for step={k}.")

    dt_eff = k * dt

    # fine start states for every jump
    q_start    = q[:, :T_cg, :]    # [n_traj, T-step, 3]
    v_start    = v[:, :T_cg, :]    # [n_traj, T-step, 3]

    # --- fine end velocities for each jump ---
    v_end = v[:, k:, :]                # v_{n+k}


    # deterministic ABOBA step with dt_eff
    _, v_det_end = aboba_deterministic_step(
        q_start, v_start,
        dt_eff=dt_eff,
        Gamma=Gamma,
        mass=mass,
        minimaDist=minimaDist,
        kconstants=kconstants,
        boxsize=boxsize,
        scale=scale,
    )

    # noise that produces the END velocity
    r_end = v_end - v_det_end          # r_{n+k} in end-aligned convention

    # --- now restrict to the coarse grid: n = 0, k, 2k, ... ---
    # end indices: k, 2k, 3k, ...
    q_eff = q[:, k::k, :]
    v_eff = v[:, k::k, :]
    
    # r_end[n] corresponds to jump from n -> n+k, i.e. lands on (n+k)
    # so pick start indices 0, k, 2k, ...
    r_eff = r_end[:, 0::k, :]

    # lengths match by construction
    assert q_eff.shape == v_eff.shape == r_eff.shape

    return q_eff, v_eff, r_eff

In [4]:
systemType='bistable'

conditionedOn='piririm'

In [5]:
# datapoint = [time (1), qi (3), vi (3), ? (1), ri(3)] -- 11 dim
# for dimer, alternating between particle 1 and particle 2.

# Datasets directory
localDirectory = "/group/ag_cmb/scratch/maojrs/stochasticClosure/" + systemType + "/boxsize5/benchmark/"

# Total no. of datasets
n_datasets = 10
train_split = 0.8

# Sample simulation files randomly
fnums = np.random.choice(2500, n_datasets, replace=False)
print(np.sort(fnums))
dataset = None

for f_num in fnums:
    try:
        ds = torch.Tensor(trajectoryTools.loadTrajectory(localDirectory + "simMoriZwanzig_", f_num)).unsqueeze(0)
    except FileNotFoundError:
        print(f'File {f_num} not available.')
        continue
              
    if dataset is None:
        dataset = ds
    else:
        dataset = torch.cat((dataset, ds), dim=0)
        
n_timesteps = dataset.shape[1]

# Dataset - training data
dataset.shape, dataset.flatten(end_dim=1).shape, n_timesteps, dataset.dtype

[  19  946 1116 1310 1700 1709 1834 1859 1969 2246]


(torch.Size([10, 10000, 11]), torch.Size([100000, 11]), 10000, torch.float32)

In [6]:
# Load parameters from parameters file
parameterDictionary = analysisTools.readParameters(localDirectory + "parameters")

dt = parameterDictionary['dt']
Gamma = parameterDictionary['Gamma']
mass = parameterDictionary['mass']

# Parameters for external potential (will only acts on distinguished particles (type 1))
minimaDist = 1.5
kconstants = np.array([1.0, 1.0, 1.0])
scalefactor = 1

In [7]:
step = 1

q = dataset[..., 1:4]  # position (not used now, but may be later)
v = dataset[..., 4:7]   # velocity
r = dataset[..., 8:11]  # auxiliary var

print(q.shape, v.shape, r.shape)

torch.Size([10, 10000, 3]) torch.Size([10, 10000, 3]) torch.Size([10, 10000, 3])


In [9]:
q_eff, v_eff, r_eff = compute_r_cg_from_fine(
    q, v, k=step, parameters=parameterDictionary
)

In [10]:
q[0, step:step+5], q_eff[0, :5], r[0, step:step+5], r_eff[0, :5]

(tensor([[-2.0239, -0.7220,  1.0474],
         [-2.0310, -0.7296,  1.0441],
         [-2.0384, -0.7372,  1.0414],
         [-2.0464, -0.7436,  1.0421],
         [-2.0552, -0.7481,  1.0489]]),
 tensor([[-2.0239, -0.7220,  1.0474],
         [-2.0310, -0.7296,  1.0441],
         [-2.0384, -0.7372,  1.0414],
         [-2.0464, -0.7436,  1.0421],
         [-2.0552, -0.7481,  1.0489]]),
 tensor([[ 0.0016, -0.0051, -0.0158],
         [-0.0057, -0.0084, -0.0055],
         [-0.0093,  0.0062,  0.0299],
         [-0.0229,  0.0363,  0.1123],
         [-0.0165,  0.0391,  0.1354]]),
 tensor([[ 0.0016, -0.0051, -0.0158],
         [-0.0057, -0.0084, -0.0055],
         [-0.0093,  0.0062,  0.0299],
         [-0.0229,  0.0363,  0.1123],
         [-0.0165,  0.0391,  0.1354]]))

In [11]:
r[:, step:].flatten() - r_eff.flatten()

tensor([9.4296e-09, 6.9849e-09, 0.0000e+00,  ..., 7.4506e-09, 0.0000e+00,
        0.0000e+00])

In [13]:
# q, v, r_true: [n_traj, T, 3] from benchmark
q_1, v_1, r_cg_1 = compute_r_cg_from_fine(
    q, v,
    k=1,
    parameters=parameterDictionary
)

# r_true is [n_traj, T, 3]; r_cg_1 is [n_traj, T-1, 3] (n=0..T-2)
r_true_cut = r[:, 1:, :]  # align shapes

diff = (r_cg_1 - r_true_cut).abs()
print("max |r_cg - r_true|:", diff.max().item())
print("mean |r_cg - r_true|:", diff.mean().item())

max |r_cg - r_true|: 1.1920928955078125e-07
mean |r_cg - r_true|: 4.868224490195416e-09


In [25]:
step = 10
q_cg, v_cg, r_cg = compute_r_cg_from_fine(
    q, v,
    k=step,
    parameters=parameterDictionary
)

dt_eff = step * dt
v_next = v[:, step::step, :]  # [n_traj, T-step, 3]

# Recompute deterministic update
_, v_det = aboba_deterministic_step(
    q[:, :-step], v[:, :-step],
    dt_eff=dt_eff,
    Gamma=Gamma,
    mass=mass,
    minimaDist=minimaDist,
    kconstants=kconstants,
    boxsize=5.0,
    scale=scalefactor,
)

residual = (v_det[:, ::step] + r_cg - v_next).abs()
print("max residual:", residual.max().item())
print("mean residual:", residual.mean().item())

max residual: 2.9802322387695312e-08
mean residual: 7.604847862552333e-10


In [18]:
bsize = 5
os.environ['DATA'] = '/group/ag_cmb/scratch/maojrs/'


#parentDirectory = os.environ.get('MSMRD') + '/data/MoriZwanzig/bistable/benchmark/'
parentDirectory = os.environ['DATA'] + 'stochasticClosure/bistable/boxsize' + str(bsize)+ '/benchmark/'
fnamebase = parentDirectory + 'simMoriZwanzig_'
foldername = 'binnedData/'

# Load parameters from parameters file
parameterDictionary = analysisTools.readParameters(parentDirectory + "parameters")
# Parameters for loading continuous trajectories from files (from original simulation)
nfiles = parameterDictionary['numFiles']
dt = parameterDictionary['dt']
stride = parameterDictionary['stride']
totalTimeSteps = parameterDictionary['timesteps']
boxsize = parameterDictionary['boxsize']
boundaryType = parameterDictionary['boundaryType']

if bsize != boxsize:
    print('Requested boxsize does not match simulation')

# Load trajectory data from h5 files (only of distinguished particle)
trajs = []
print("Loading data ...")

"""here you can decide how many trajs you need"""
for i in range(200):
    traj = trajectoryTools.loadTrajectory(fnamebase, i)
    trajs.append(traj)
    sys.stdout.write("File " + str(i+1) + " of " + str(nfiles) + " done." + "\r")
print("\nAll data loaded.")
print(' ')

Loading data ...
File 200 of 2500 done.
All data loaded.
 


In [19]:
trajs_arr = np.array(trajs)
data_x = trajs_arr[:,:,1:4]
data_v = trajs_arr[:,:,4:7]
data_r = trajs_arr[:,:,8:]
parameters =parameterDictionary
print(parameters)

# Parameters for external potential (will only acts on distinguished particles (type 1))
minimaDist = 1.5
kconstants = np.array([1.0, 1.0, 1.0])
scalefactor = 1

# Extract basic parameters
dt = parameters['dt']
Gamma = parameters['Gamma']
mass =  parameters['mass']
KbT = parameters['KbT']
boxsize = parameters['boxsize']
boundaryType = parameters['boundaryType']

if bsize != boxsize:
    print('Requested boxsize does not match simulation')

{'numFiles': 2500, 'numParticles': 501, 'dt': 0.05, 'bodytype': 'point', 'Gamma': 0.3, 'sigma': 0.44544935907016964, 'KbT': 1, 'mass': 54.0, 'timesteps': 10000, 'stride': 1, 'trajtype': 'moriZwanzigVelocity', 'boxsize': 5, 'boundaryType': 'periodic', 'equilibrationSteps': 5000}


In [20]:
def integrateA(position, velocity,dt,t_step):
    return position + dt*t_step *velocity

def enforce_periodic_boundary(position, boxsize):
    """
    Apply periodic boundary conditions.
    - position: a list/array of coordinates (e.g., [x, y, z])
    - boxsize: a list/array of box lengths (same dimension)
    Returns a NEW wrapped position.
    """
    new_position = position.copy()  # 不修改原输入

    for j in range(len(position)):
        half_box = boxsize[j] / 2

        if new_position[j] >= half_box:
            new_position[j] -= boxsize[j]
        elif new_position[j] <= -half_box:
            new_position[j] += boxsize[j]

    return new_position

def bistable_force_from_position(position,  scale=1.0):
    """
    Compute the force from the bistable potential directly from a 3D position.
    
    position: 3D coordinate (list or array)
    minimaDist: distance of the two minima (float)
    kconstants: [k1, k2, k3]
    scale: scale factor for the potential

    Returns:
        A 3D numpy array representing the force.
    """

    x = np.array(position)
    k1, k2, k3 = kconstants

    force = np.zeros(3)

    # Force in x from d/dx of k1 * (1 - (x/minDist)^2)^2
    force[0] = - k1 * 4 * x[0] * (x[0]**2 - minimaDist**2) / minimaDist**4

    # Harmonic terms
    force[1] = - k2 * 2 * x[1]
    force[2] = - k3 * 2 * x[2]

    return scale * force

def integrate_BOB_step(vn, vn1, force, dt,t_step):

    # exp term
    expterm = np.exp(-dt*t_step * Gamma / mass)

    # friction + deterministic force contribution
    frictionForceTerm = vn * expterm
    frictionForceTerm += (1 + expterm) * force * dt*t_step / (2 * mass)

    interactionNoiseTerm = vn1 - frictionForceTerm

    # new velocity would be (not returned per your request):
    # v_next = frictionForceTerm + interactionNoiseTerm

    return interactionNoiseTerm

In [21]:
def calculate_r_t_step(data_r,t_step):
    r_t_step =[]
    for j in tqdm(range(len(data_r)), desc="Calculating r_t_step"):
        r_value = []
        for i in range(len(data_r[0])-t_step):
            position = data_x[j,i,:]
            velocity = data_v[j,i,:]
            vn1 =data_v[j,i+1,:]
            position = integrateA(position,velocity,dt/2,t_step)
            position = enforce_periodic_boundary(position,[5.0,5.0,5.0])
            force = bistable_force_from_position(position)
            rnp1 = integrate_BOB_step(velocity,vn1,force,dt,t_step)
            r_value.append(rnp1)
        r_t_step.append(r_value)
    
    return r_t_step
    

In [22]:
r_t_step = np.array(calculate_r_t_step(data_r,t_step = 1))
print(r_t_step.shape)


Calculating r_t_step:   4%|█▋                                             | 7/200 [00:10<04:59,  1.55s/it]

KeyboardInterrupt



In [8]:
mask = []

for k in range(len(r_t_step)):
    mask.append(np.allclose(r_t_step[k, :, :], data_r[k, 1:, :]))
print(mask)

[False, True, True, False, False, True, True, True, True, False]


In [9]:
r_t_step_good = r_t_step[mask]
data_r_good = data_r[mask]

In [10]:
print(r_t_step_good.shape)
print(data_r_good.shape)
np.allclose(r_t_step_good[:, :, :], data_r_good[:, 1:, :])

(6, 9999, 3)
(6, 10000, 3)


True

In [11]:
num = 1900
t_step = 1
velocity = data_v[0,num,:]
vn1 = data_v[0,num+1,:]
position = data_x[0,num,:]
position = integrateA(position,velocity,dt/2,t_step)
position = enforce_periodic_boundary(position,[5.0,5.0,5.0])
force = bistable_force_from_position(position)
rnp1 = integrate_BOB_step(velocity,vn1,force,dt,t_step)

In [12]:
print(rnp1)

[ 1.68989706e-02 -5.36055934e-05  5.82440296e-03]


In [13]:
print(data_r[0,num+1,:])

[ 2.26880327e-03 -5.36055934e-05  5.82440296e-03]
